In [1]:
def generate_signals(macro, prices):
    # Signal: Buy SPY when yield curve (10y-2y) > 1%, move to cash when < 0, else neutral
    spy_signal = (
        macro['spread_10y2y'].gt(1).astype(float) - macro['spread_10y2y'].lt(0).astype(float)
    ).shift(1).fillna(0)

    # Resample to monthly, align to prices index
    signals = pd.DataFrame(index=macro.index)
    signals['SPY'] = spy_signal
    monthly = signals.resample('ME').last()  # Monthly endpoints
    out = monthly.reindex(prices.index, method='ffill').fillna(0)
    # Zero out signals for days not present in macro
    out = out.where(signals.reindex(prices.index).notnull(), 0)
    # Expand to all ETFs, default to 0
    for ticker in prices.columns:
        if ticker != 'SPY':
            out[ticker] = 0.0
    # Reorder columns to match prices
    return out[prices.columns].astype(float).fillna(0)